# Lab 10 — Enhancement failure modes: did I recover the signal or manufacture one?

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 10 — §10.3 (the √N law and its assumptions), §10.5 (LMS adaptive filtering), §10.7 (ICA/PCA unmixing), §10.8 (method-selection guidelines).

**Biomedical question.** Did my enhancement recover the *true* signal, or manufacture a plausible-looking one?
**Task type (§1.8).** Denoising / enhancement — with honest verification against ground truth.
**Information that must be preserved.** the DESIRED source — and the ability to *prove* it survived the cleanup.
**Main assumptions.** each method's precondition holds: averaging needs a *repeating* signal in *uncorrelated* noise; LMS needs a reference *correlated with the artifact but not with the desired source*; ICA needs *independent, non-Gaussian* sources.
**Primary diagnostic.** compare the enhanced output against the KNOWN truth — SNR-gain vs the √N law, correlation of the LMS output with the true clean source, and best-match correlation of the ICA components to the true sources.
**Transfer challenge.** each check here leans on truth we happen to know; on real data you rarely do — what reference-free surrogate would you trust instead?

*Self-contained: seeded synthetic signals (`np.random.default_rng(2013)`), numpy / scipy, and `sklearn.decomposition.FastICA`. No `bsp`, no data files, no I/O. Runs fully offline in well under a minute. Theme (§1.8): a clean-LOOKING output is not evidence — enhancement can **manufacture** a plausible signal, so every method is judged against the very thing it was supposed to preserve.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab10_enhancement_failure_modes/lab10_enhancement_failure_modes.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab10_enhancement_failure_modes.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab10_enhancement_failure_modes.ipynb)

In [ ]:
# --- shared setup (reproducible; fully offline) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
from sklearn.decomposition import FastICA
from itertools import permutations
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

## 1. √N averaging — the honest workhorse

Ensemble averaging is the safest enhancement there is: line up `N` repeats of a *repeating* signal (an evoked potential locked to a stimulus) and average. If the noise is **uncorrelated** trial-to-trial while the signal repeats, the noise amplitude shrinks like 1/√N, so the **power SNR rises ~+3 dB per doubling of N** — the √N law. We build a known template, bury it in noise, average for N = 1, 4, 16, 64, 256, and check the *measured* gain against the prediction 10·log₁₀(N).

In [ ]:
# TODO build a known evoked-potential template, make N noisy repeats (template + UNCORRELATED
#   Gaussian noise), ensemble-average for each N in Ns, and MEASURE the post-average power SNR as
#   signal_power / residual_power with residual = (ensemble average - template), i.e. the leftover
#   noise measured against the TRUTH. Average that residual power over M fresh ensembles so the curve
#   is smooth. Report the SNR-GAIN relative to N=1 against the √N prediction 10*log10(N).
fs = 500
t = np.arange(int(0.6*fs))/fs                       # a 600 ms evoked-potential epoch
template = -1.0*np.exp(-((t-0.10)/0.020)**2) + 0.7*np.exp(-((t-0.20)/0.035)**2)   # N100 / P200-like
noise_sd = 3.0                                       # single-trial noise DWARFS the evoked response
Ns = [1, 4, 16, 64, 256]
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the assumption is UNCORRELATED noise + a REPEATING signal. What happens to this gain if
#   the "noise" is actually a time-locked artifact (something correlated with the stimulus)?

## 2. LMS adaptive cancellation — and did the desired source survive?

Now the harder case. The primary channel carries a desired physiological source **plus a motion artifact that is correlated with a reference channel** (an accelerometer strapped to the body). An **LMS adaptive filter** learns the coupling path from the reference and subtracts its prediction, leaving an error signal that should be the clean source. The output will *look* clean — but looking clean is not the test. Because this is a lab we KNOW the true source, so we can run the real test: **does the LMS output still correlate with the truth?**

In [ ]:
# TODO (a) build a desired source `s_clean`, an accelerometer-like BROADBAND reference `ref`, and a
#   motion `artifact` that is a causal FIR coupling of the reference; the recorded `primary` = source +
#   artifact, with the artifact dominating. (b) Run a normalized-LMS filter (L taps, step mu) that
#   predicts the artifact from `ref`; the ERROR signal `e` is the cleaned output. (c) VERIFY: report
#   corr(primary, truth) BEFORE and corr(e, truth) AFTER — a clean-looking `e` is worthless unless it
#   still matches the desired source we set out to preserve.
fs2 = 250
tt = np.arange(8*fs2)/fs2
s_clean = np.sin(2*np.pi*7*tt) + 0.5*np.sin(2*np.pi*13*tt + 0.6)     # desired source (7 & 13 Hz)
ref = 0.6*np.sin(2*np.pi*0.9*tt) + rng.standard_normal(tt.size)      # accelerometer: slow sway + broadband motion
ref = ref/np.std(ref)
h_true = np.array([0.9, -0.6, 0.35, 0.15])                          # UNKNOWN body->electrode coupling path
artifact = sig.lfilter(h_true, [1.0], ref)                          # motion couples in via a causal FIR
primary = s_clean + 2.5*artifact                                    # the raw recorded channel (artifact dominates)
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the output looks clean either way. Which single number tells you it is the RECOVERED
#   source and not a manufactured one? What could you check if you had NO ground-truth source at all?

## 3. ICA — powerful, and unstable at the edges

Two independent sources are mixed into two channels and `FastICA` unmixes them. It works remarkably well — but it recovers each source only **up to sign, scale, and order (permutation)**: the algorithm has no way to know which component is *source 1*, which way is up, or how loud it was. We recover the sources, report the best-match correlation to truth, and then show the instability directly — **re-seeding FastICA flips a component's sign** while the match stays near-perfect. A component that comes back inverted and reordered is *not* automatically a physiological signal.

In [ ]:
# TODO (a) build two INDEPENDENT non-Gaussian sources and mix them with a 2x2 matrix A into 2 channels.
#   (b) run FastICA (n_components=2) to recover them. (c) report the BEST-MATCH |correlation| to the
#   true sources after resolving the permutation. (d) DEMONSTRATE the sign/scale/permutation ambiguity:
#   run FastICA again with a DIFFERENT random_state and show a component returns with flipped sign /
#   swapped order while |corr| stays ~1.
N3 = 2000
tt3 = np.linspace(0, 8, N3)
S = np.c_[np.sin(2*np.pi*2.0*tt3), sig.sawtooth(2*np.pi*0.7*tt3)]    # a sine and a sawtooth (independent, non-Gaussian)
S = (S - S.mean(0)) / S.std(0)
A = np.array([[1.0, 0.7], [0.6, 1.2]])                              # the unknown mixing matrix
Xmix = S @ A.T                                                      # two recorded channels
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: |corr| ~ 1 says the sources were recovered, but sign / scale / order were NOT fixed. Why
#   does that forbid reading a component's polarity or amplitude as physiology without an external anchor?

## Live sanity check

One cell ties the three claims together, all computed live: the averaging gain must track the √N law; the LMS output must still correlate with the TRUE source (> 0.9) and beat the raw channel; and ICA must recover the sources up to sign/permutation (best-match |corr| near 1). If any assertion fails, an *enhancement* has manufactured something rather than recovered it.

In [ ]:
# --- live self-checks (every number was computed in the cells above) ---
assert np.max(np.abs(gain_meas - gain_pred)) < 0.5,            "√N averaging gain drifted from the prediction"
assert abs(np.mean(np.diff(gain_meas)/2) - 3.0103) < 0.3,     "averaging gain is not ~+3 dB per doubling"
assert c_after > 0.9 and c_after > c_before + 0.3,            "LMS output does not match the TRUE source"
assert np.min(np.abs(signed0)) > 0.95,                        "ICA did not recover the sources (up to sign/perm)"
print("√N gain vs prediction (dB):", {N: round(g, 2) for N, g in zip(Ns, gain_meas)})
print(f"√N law     : measured {np.mean(np.diff(gain_meas)/2):.2f} dB/doubling  vs predicted 3.01, "
      f"max |error| {np.max(np.abs(gain_meas-gain_pred)):.3f} dB")
print(f"LMS        : corr to truth {c_before:.3f} -> {c_after:.3f}   (desired source survived: {c_after > 0.9})")
print(f"ICA        : best-match |corr| {np.round(np.abs(signed0), 3)}, permutation {perm0}, "
      f"sign-flip across seeds on comp {flipped if flipped else 'none (this pair)'}")
print("\nALL SANITY CHECKS PASSED — each enhancement was verified against the truth it had to preserve.")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the reasoning, not hitting any particular number.

1. **Which check was the real evidence?** For each method name the ONE quantity that proved the desired source *survived* (not merely that the output looked clean), and say what you would use as a stand-in when you do NOT have ground truth — e.g. a held-out no-stimulus / surrogate control, split-half reproducibility, or an independent physiological correlate.
2. **Precondition failure.** Pick one method and describe an input where it *manufactures* a signal: averaging when the noise is time-locked to the stimulus (it survives the average and *looks* like signal); LMS when the reference is also correlated with the DESIRED source (it cancels the very thing you wanted); ICA when you read a component's sign / scale / order as physiology. Which preserved-information requirement (§1.8) does each one break?
3. **Transfer.** Your LMS canceller worked with a reference that was **uncorrelated** with the desired source, even though its broadband content overlapped the source's frequency bands. On a new recording the artifact overlaps the source band. What would you re-measure before trusting the cleaned output there?

**Rule out (name the wrong move).** State the classic error plainly: *trusting a clean-LOOKING LMS or ICA output without checking that the DESIRED source survived*, or *reading ICA components as ground-truth physiology* when their sign, scale and order are arbitrary. Name the requirement it breaks: it violates the **§1.8** rule that enhancement must **preserve the desired source AND let you prove it survived** — a plausible-looking output that was never verified against the thing it had to preserve is a *manufactured* signal, not a recovered one.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic signals; illustrative numbers. The lesson is the method: enhancement is trustworthy only when the desired source is checked against something you can defend — averaging, adaptive cancellation and ICA all pass here only because we verified each against the truth.*